In [ ]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="llama3.2:latest")
print('LLM is ready')

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage

my_chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chat_history = []

In [17]:
def ask(user_text: str, history:str):
    global chat_history
    my_chat_prompt = my_chat_prompt_template.format_messages(chat_history=chat_history, input=user_text)
    #print(f"chat prompt: {my_chat_prompt}")
    response = model.invoke(my_chat_prompt)
    chat_history.append(HumanMessage(content=user_text))
    chat_history.append(SystemMessage(response.content))
    print(response)
    return [{"role": "assistant", "content": response.content}]



In [ ]:
while True:
    user_text = input("Enter a prompt or type quit to exit")
    if user_text.lower() == 'quit':
        break
    response = ask(user_text)
    print(response)


In [ ]:
import gradio as gr

# Create Gradio Chat Interface
with gr.Blocks() as demo:
    gr.Markdown("## 💬 Chat with Ollama Model")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Type your message here...")
    clear = gr.Button("Clear Chat")

    msg.submit(ask, [msg, chatbot], chatbot)
    clear.click(lambda: None, None, chatbot, queue=False)

# Run the Gradio app
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7865)